# Focus-case financial tables

Generate presentation-ready **base-case financial tables** from the saved workbook caches. The workbook is read-only; P13 USD-converted values are returned to EUR. These tables are plotted inside this notebook with Pillow and contain explicit assumptions, not observed TotalEnergies results. Existing scenario J-curves and the original workbook are not modified.


In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import openpyxl

# Read workbook caches only. Never save or modify the source workbook.
root = Path.cwd().resolve()
while not (root / 'work/finance/portfolio_model_14_revised.xlsx').exists():
    if root == root.parent:
        raise FileNotFoundError('Run inside the repository')
    root = root.parent
workbook = openpyxl.load_workbook(root / 'work/finance/portfolio_model_14_revised.xlsx', data_only=True, read_only=True)
build, inputs = workbook['Cash flow build'], workbook['Inputs']
out = root / 'work/projects/assets'
out.mkdir(parents=True, exist_ok=True)
fontdir = Path('C:/Windows/Fonts')
def font(size, bold=False):
    return ImageFont.truetype(str(fontdir / ('arialbd.ttf' if bold else 'arial.ttf')), size)

def render_case(index, ident, title, currency, caution):
    # Workbook 'Cash flow build' is USD for all rows; return P13 to EUR.
    r = 2 + 7 * (3 * index + 1)
    if build.cell(r, 1).value != ident + ' / Base':
        raise ValueError('Unexpected project row: ' + ident)
    fx = float(inputs.cell(index + 2, 5).value)
    launch = float(inputs.cell(index + 2, 9).value)
    get = lambda offset: [float(build.cell(r + offset, 2 + y).value or 0.0) / fx for y in range(6)]
    gross, costs, capex, tax, net, cum = [get(x) for x in range(1, 7)]
    recurring = costs.copy(); recurring[1] -= launch
    rows = [
        ('Gross incremental benefit', gross),
        ('Recurring OPEX', [-x for x in recurring]),
        ('One-time launch', [0.0, -launch, 0.0, 0.0, 0.0, 0.0]),
        ('Capital spending', [-x for x in capex]),
        ('Cash tax', [-x for x in tax]),
        ('Net incremental cash', net),
        ('Cumulative net cash', cum),
    ]
    for y in range(6):
        if abs(sum(v[y] for _, v in rows[:5]) - net[y]) > 1e-8:
            raise ValueError('Cash reconciliation failed for %s Y%d' % (ident, y))
    width, height = 2400, 1050
    im = Image.new('RGB', (width, height), '#ffffff')
    d = ImageDraw.Draw(im)
    title_font, sub_font = font(60, True), font(31)
    cell_font, bold_font, small_font = font(34), font(34, True), font(27)
    d.text((65, 38), ident + ' | ' + title, font=title_font, fill='#153455')
    d.text((68, 128), 'BASE SCENARIO  |  ' + currency + ' million  |  Y0–Y5  |  post-tax incremental cash', font=sub_font, fill='#465870')
    x0, x1 = 65, 2335
    first_width = 720
    colw = (x1 - x0 - first_width) / 6
    top, rowh = 220, 82
    d.rounded_rectangle((x0, top, x1, top + rowh), radius=9, fill='#17365d')
    d.text((x0 + 20, top + 22), 'Cash-flow driver', font=bold_font, fill='white')
    for y in range(6):
        cx = x0 + first_width + colw * (y + .5)
        d.text((cx, top + 41), 'Y%d' % y, font=bold_font, fill='white', anchor='mm')
    for i, (label, values) in enumerate(rows):
        y0 = top + rowh * (i + 1)
        fill = '#e4edf6' if i >= 5 else ('#f5f8fb' if i % 2 == 0 else '#ffffff')
        d.rectangle((x0, y0, x1, y0 + rowh), fill=fill)
        d.line((x0, y0 + rowh, x1, y0 + rowh), fill='#d6dfe8', width=2)
        f = bold_font if i >= 5 else cell_font
        d.text((x0 + 20, y0 + 22), label, font=f, fill='#172a40')
        for j, number in enumerate(values):
            if abs(number) < .0005: number = 0.0
            cx = x0 + first_width + colw * (j + .5)
            d.text((cx, y0 + 41), ('%.3f' % number), font=f, fill=('#a52735' if number < 0 else '#172a40'), anchor='mm')
    d.text((68, 898), 'Figures are assumptions, not TotalEnergies actuals. Three-decimal rounding; no terminal value.', font=small_font, fill='#4b5c71')
    d.text((68, 945), caution, font=small_font, fill='#4b5c71')
    filename = 'notebook_' + ident.lower() + '_cash_table.png'
    im.save(out / filename, dpi=(200, 200))
    print(ident, 'saved', filename, 'final cumulative', round(cum[-1], 6))

render_case(0, 'P01', 'Refining predictive maintenance', 'USD', 'Cash tax and capitalisation are assumptions. Source: workbook base scenario; hurdle 9.5%.')
render_case(2, 'P03', 'Deepwater drilling advisory', 'USD', 'Cash savings require actual rig-release or invoice rights. Source: workbook base; hurdle 11%.')
render_case(10, 'P11', 'Customer flexibility option', 'USD', 'Conditional commercial illustration, not guaranteed revenue or H3 payback. Hurdle 11%.')
render_case(12, 'P13', 'Retail demand pilot', 'EUR', 'Hypothetical 100-station pilot; EUR values restored from workbook USD translation. Hurdle 9%.')
workbook.close()
